In [7]:
from google.colab import files

uploaded = files.upload()   # choose appointments.csv in the dialog
print("Uploaded:", list(uploaded.keys()))


Saving appointments.csv to appointments (2).csv
Uploaded: ['appointments (2).csv']


In [8]:
import csv
import json

CSV_PATH = "appointments.csv"
REQUIRED = ["age", "fee_inr", "prior_no_shows", "distance_km"]

# ---- 1. Open and read with csv.DictReader ----
with open(CSV_PATH, newline="") as f:
    reader = csv.DictReader(f)
    header = reader.fieldnames
    rows = list(reader)

print("Header   :", header)
print("Row count:", len(rows))

# ---- Check the file has the required columns ----
missing = [col for col in REQUIRED if col not in (header or [])]
if missing:
    print(f"\nWrong file: missing columns {missing}")
    print("This looks like a different CSV. Upload the real appointments.csv "
          "(or create the sample file) and run this cell again.")
else:
    # ---- 2. parse_row ----
    def parse_row(row):
        """Convert numeric fields and return the converted dictionary."""
        converted = dict(row)  # copy so the original row is untouched
        converted["age"] = int(row["age"])
        converted["fee_inr"] = int(row["fee_inr"])
        converted["prior_no_shows"] = int(row["prior_no_shows"])
        converted["distance_km"] = float(row["distance_km"])
        return converted

    # ---- 3. Loop over every row ----
    accepted = []
    rejected = []

    for row in rows:
        try:
            accepted.append(parse_row(row))
        except (ValueError, TypeError):
            # ValueError: bad text like "N/A" or ""
            # TypeError: None value when a row has fewer fields than the header
            rejected.append(row)

    print("Accepted :", len(accepted))
    print("Rejected :", len(rejected))

    # ---- 4. Save rejected rows as JSON ----
    with open("rejected_rows.json", "w") as f:
        json.dump(rejected, f, indent=2)

    print("Saved rejected rows to rejected_rows.json")

Header   : ['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude']
Row count: 8

Wrong file: missing columns ['age', 'fee_inr', 'prior_no_shows', 'distance_km']
This looks like a different CSV. Upload the real appointments.csv (or create the sample file) and run this cell again.


In [9]:
# The error occurred because "appointments.csv" already existed,
# so Colab saved your new upload as "appointments (2).csv".
# Let's run the parser with the newly uploaded file!

import csv
import json

# Pointing to the newly uploaded appointments file
CSV_PATH = "appointments (2).csv"
REQUIRED = ["age", "fee_inr", "prior_no_shows", "distance_km"]

# ---- 1. Open and read with csv.DictReader ----
with open(CSV_PATH, newline="") as f:
    reader = csv.DictReader(f)
    header = reader.fieldnames
    rows = list(reader)

print("Header   :", header)
print("Row count:", len(rows))

# ---- Check the file has the required columns ----
missing = [col for col in REQUIRED if col not in (header or [])]
if missing:
    print(f"\nWrong file: missing columns {missing}")
else:
    # ---- 2. parse_row ----
    def parse_row(row):
        """Convert numeric fields and return the converted dictionary."""
        converted = dict(row)  # copy so the original row is untouched
        converted["age"] = int(row["age"])
        converted["fee_inr"] = int(row["fee_inr"])
        converted["prior_no_shows"] = int(row["prior_no_shows"])
        converted["distance_km"] = float(row["distance_km"])
        return converted

    # ---- 3. Loop over every row ----
    accepted = []
    rejected = []

    for row in rows:
        try:
            accepted.append(parse_row(row))
        except (ValueError, TypeError):
            rejected.append(row)

    print("Accepted :", len(accepted))
    print("Rejected :", len(rejected))

    # ---- 4. Save rejected rows as JSON ----
    with open("rejected_rows.json", "w") as f:
        json.dump(rejected, f, indent=2)

    print("Saved rejected rows to rejected_rows.json")

Header   : ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Row count: 1020
Accepted : 988
Rejected : 32
Saved rejected rows to rejected_rows.json


In [11]:
import pandas as pd

CSV_PATH = "appointments (2).csv"

# ---- Load the original file ----
df = pd.read_csv(CSV_PATH)

# ---- Initial inspection ----
print("Shape:", df.shape)
print("\nInfo:")
df.info()
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

print("\n--- Cleaning steps ---")
print("Start             :", len(df))

# (a) Drop exact duplicate rows
df = df.drop_duplicates()
print("After (a) dedupe  :", len(df))

# (b) Clean booking_channel
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("After (b) channel :", len(df))

# (c) Convert dates
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("After (c) dates   :", len(df))

# (d) Drop rows with missing appt_date (after conversion) or missing status
df = df.dropna(subset=["appt_date", "status"])
print("After (d) dropna  :", len(df))

Shape: (1020, 14)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing values per column:
appointment_id      0
patient_id       

In [12]:
import numpy as np
import pandas as pd

# Continues from the cleaned `df` produced in the previous step
# (age, distance_km, fee_inr columns must exist).

# ---- (a) Invalid age -> missing ----
invalid_age = (df["age"] < 1) | (df["age"] > 100)
print("Invalid ages set to missing:", int(invalid_age.sum()))
df.loc[invalid_age, "age"] = np.nan

# ---- (b) Fill gaps with the median ----
age_median = df["age"].median()
dist_median = df["distance_km"].median()

df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(dist_median)

print("Missing age after fill        :", int(df["age"].isnull().sum()))
print("Missing distance_km after fill:", int(df["distance_km"].isnull().sum()))

# ---- (c) IQR fences and capping with np.clip ----
for col in ["distance_km", "fee_inr"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    n_capped = int(((df[col] < lower) | (df[col] > upper)).sum())
    df[col] = np.clip(df[col], lower, upper)

    print(f"{col}: Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}, "
          f"fences=[{lower:.2f}, {upper:.2f}], capped={n_capped}")

# ---- (d) Rounding ----
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)

print("\nFinal dtypes:")
print(df[["age", "distance_km", "fee_inr"]].dtypes)
print("Row count:", len(df))

Invalid ages set to missing: 5
Missing age after fill        : 0
Missing distance_km after fill: 0
distance_km: Q1=4.62, Q3=13.00, IQR=8.38, fences=[-7.94, 25.56], capped=26
fee_inr: Q1=500.00, Q3=787.50, IQR=287.50, fences=[68.75, 1218.75], capped=3

Final dtypes:
age              int64
distance_km    float64
fee_inr          int64
dtype: object
Row count: 990


In [13]:
import numpy as np
import pandas as pd

# Continues from the cleaned `df` (dates converted, age/distance_km/fee_inr cleaned).

# ---- New columns ----
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

print("lead_days range:", df["lead_days"].min(), "to", df["lead_days"].max())
print("no_show counts:\n", df["no_show"].value_counts())

# ---- Feature matrix ----
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()
print("\nX shape:", X.shape)

# (a) Mean of each column
col_means = X.mean(axis=0)
for name, m in zip(["age", "distance_km", "fee_inr", "lead_days"], col_means):
    print(f"Mean {name:12s}: {m:.2f}")

# (b) Rows with lead_days > 14 (lead_days is column index 3)
mask = X[:, 3] > 14
print(f"\nRows with lead_days > 14: {mask.sum()} of {len(X)} "
      f"({mask.mean():.1%})")

# (c) z-scores of distance_km (column index 1)
x = X[:, 1]
z = (x - np.mean(x)) / np.std(x)
print(f"Rows with |z| > 2 on distance_km: {(np.abs(z) > 2).sum()}")

lead_days range: 1 to 45
no_show counts:
 no_show
0    772
1    218
Name: count, dtype: int64

X shape: (990, 4)
Mean age         : 48.17
Mean distance_km : 9.50
Mean fee_inr     : 635.71
Mean lead_days   : 11.82

Rows with lead_days > 14: 296 of 990 (29.9%)
Rows with |z| > 2 on distance_km: 53


In [14]:
from google.colab import files

uploaded = files.upload()   # choose appointments.csv in the dialog
print("Uploaded:", list(uploaded.keys()))

Saving clinics.csv to clinics.csv
Uploaded: ['clinics.csv']


In [15]:
import requests
import pandas as pd

CLINICS_PATH = "clinics.csv"   # the file with clinic_id, clinic_name, city, tier, latitude, longitude
URL = "https://api.open-meteo.com/v1/forecast"

clinics = pd.read_csv(CLINICS_PATH)
print("Clinics:", len(clinics))


def fetch_temp(lat, lon):
    """Return (temp_c, reason). temp_c is None if anything goes wrong."""
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "temperature_2m",
        "forecast_days": 1,
    }
    try:
        resp = requests.get(URL, params=params, timeout=10)
    except requests.exceptions.RequestException as e:
        return None, f"request error: {e}"

    if resp.status_code != 200:
        return None, f"status code {resp.status_code}"

    try:
        data = resp.json()
        # First hourly value = midnight-UTC reading (a stand-in, not current weather)
        temp = data["hourly"]["temperature_2m"][0]
    except (ValueError, KeyError, IndexError, TypeError) as e:
        return None, f"'hourly' data missing or malformed ({type(e).__name__})"

    if temp is None:
        return None, "temperature value is null in the reply"
    return temp, "ok"


temps = []
for _, row in clinics.iterrows():
    temp, reason = fetch_temp(row["latitude"], row["longitude"])

    if temp is None:                      # retry this clinic once
        temp, reason = fetch_temp(row["latitude"], row["longitude"])

    if temp is None:
        print(f"FAILED: {row['clinic_name']} -> {reason}")
    else:
        print(f"OK    : {row['clinic_name']} -> {temp} °C")

    temps.append(temp)                    # None if it still failed

clinics["temp_c"] = temps
clinics.to_csv("clinics_enriched.csv", index=False)

print("\nSaved clinics_enriched.csv")
print(clinics.head(8))
print("Missing temp_c:", clinics["temp_c"].isnull().sum())

Clinics: 8
OK    : Kothrud Family Clinic -> 21.0 °C
OK    : Hadapsar Care Centre -> 21.5 °C
OK    : Sitabuldi Health Point -> 23.8 °C
OK    : Malviya Nagar Clinic -> 19.9 °C
OK    : Vaishali Nagar Clinic -> 19.9 °C
OK    : Edappally Medical Centre -> 26.4 °C
OK    : Vijay Nagar Clinic -> 20.2 °C
OK    : Gomti Nagar Care Point -> 19.9 °C

Saved clinics_enriched.csv
  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882   
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150   
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873   
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085   
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937   
7

In [16]:
import pandas as pd

# Continues from the cleaned `df` (dates converted, age/distance_km/fee_inr cleaned,
# lead_days and no_show created).

COLS = ["appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
        "age", "gender", "distance_km", "fee_inr", "booking_channel",
        "visit_type", "reminder_sent", "prior_no_shows", "status",
        "lead_days", "no_show"]

# ---- 1. Sort, format dates, select the 16 columns ----
out = df.sort_values("appointment_id").copy()
out["booked_on"] = out["booked_on"].dt.strftime("%Y-%m-%d")
out["appt_date"] = out["appt_date"].dt.strftime("%Y-%m-%d")
out = out[COLS]

# ---- 2. Checks before exporting ----
print("Shape:", out.shape)
print("appointment_id unique:", out["appointment_id"].is_unique)
print("\nMissing values per column:")
print(out.isnull().sum())

assert out["appointment_id"].is_unique, "Duplicate appointment_id found"
assert out.isnull().sum().sum() == 0, "Missing values found - fix them before exporting"
assert list(out.columns) == COLS and len(COLS) == 16

# ---- 3. Save appointments_clean.csv ----
out.to_csv("appointments_clean.csv", index=False)
print("\nSaved appointments_clean.csv")

# ---- 4. Load clinics_enriched.csv ----
clinics = pd.read_csv("clinics_enriched.csv")

# Safety check: f-string SQL is only OK because no value contains an apostrophe
for name, frame in [("appointments", out), ("clinics", clinics)]:
    for col in frame.select_dtypes(include="object").columns:
        bad = frame[col].astype(str).str.contains("'").sum()
        assert bad == 0, f"{name}.{col} has {bad} value(s) with an apostrophe"


def sql_val(v):
    """Text -> 'quoted', missing -> NULL (unquoted), numbers -> unquoted."""
    if pd.isnull(v):
        return "NULL"
    if isinstance(v, str):
        return f"'{v}'"
    return f"{v}"


# ---- 5. Write load_data.sql (clinics first, then appointments) ----
with open("load_data.sql", "w", encoding="utf-8") as f:
    for row in clinics.itertuples(index=False):
        values = ",".join(sql_val(v) for v in row)
        f.write(f"INSERT INTO clinics VALUES ({values});\n")

    for row in out.itertuples(index=False):
        values = ",".join(sql_val(v) for v in row)
        f.write(f"INSERT INTO appointments VALUES ({values});\n")

print(f"Wrote load_data.sql: {len(clinics)} clinics + {len(out)} appointments")

# ---- 6. Preview ----
with open("load_data.sql", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i in (0, len(clinics)):      # first clinic line, first appointment line
            print(line.strip())

Shape: (990, 16)
appointment_id unique: True

Missing values per column:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64

Saved appointments_clean.csv
Wrote load_data.sql: 8 clinics + 990 appointments
INSERT INTO clinics VALUES ('C01','Kothrud Family Clinic','Pune','Metro',18.5074,73.8077,21.0);
INSERT INTO appointments VALUES ('A0001','P509','C05','2025-01-24','2025-01-25',60,'F',18.0,850,'Phone','Consultation','Yes',1,'No-Show',1,1);
